# 04 Structured Outputs & Constrained Decoding

**Real-World Scenario**: Automated Candidate Resume & Job Application Entity Extractor.

This notebook demonstrates Pydantic Schema enforcement, LangChain `.with_structured_output()`, and a **PyTorch `LogitProcessor` simulation** of Finite State Automata (FSA) logit bias masking ($-\infty$).

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from root .env file
load_dotenv()

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))


## Step 2: Defining Pydantic Data Models & LangChain Native Structured Output

In [ ]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

class WorkExperience(BaseModel):
    company: str = Field(description="Company name")
    role: str = Field(description="Job title")
    years: float = Field(description="Years in role")

class CandidateProfile(BaseModel):
    full_name: str = Field(description="Full legal candidate name")
    primary_skills: list[str] = Field(description="List of top technical skills")
    experience: list[WorkExperience] = Field(description="Work experience list")

if os.getenv("OPENAI_API_KEY"):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    structured_llm = llm.with_structured_output(CandidateProfile)
    
    resume_text = "Alice Smith has 4 years of experience as a Senior AI Engineer at Nvidia building PyTorch models. Skills: Python, PyTorch, C++."
    result = structured_llm.invoke(f"Extract candidate profile:
{resume_text}")
    print("=== Parsed Pydantic Object Output ===")
    print("Candidate Name:", result.full_name)
    print("Primary Skills:", result.primary_skills)
    print("Work Experience:", result.experience)


## Step 3: PyTorch `LogitProcessor` Constrained Decoding Simulation (FSA Logit Bias Masking)

In [ ]:
import torch
from torch.nn.functional import softmax

# Simulated 5-token vocabulary logits: ['"name"', '"age"', 'class', 'SELECT', '{']
raw_logits = torch.tensor([[4.0, 3.5, 5.0, 2.0, 1.0]])

# FSA State Machine determines allowed valid JSON key tokens: IDs [0, 1]
valid_token_ids = [0, 1]

# Construct Logit Mask: non-valid tokens set to -inf
logit_mask = torch.full_like(raw_logits, float('-inf'))
logit_mask[0, valid_token_ids] = 0.0

constrained_logits = raw_logits + logit_mask
unconstrained_probs = softmax(raw_logits, dim=-1)
constrained_probs = softmax(constrained_logits, dim=-1)

print("=== Logit Bias Masking Simulation Results ===")
print("Unconstrained Softmax Probabilities:
", unconstrained_probs[0].tolist())
print("Constrained Softmax Probabilities (-inf Mask):
", constrained_probs[0].tolist())
print(f"Syntax Error Risk: {unconstrained_probs[0, 2:].sum().item()*100:.1f}% -> {constrained_probs[0, 2:].sum().item()*100:.1f}%")
